# NLP Hackathon — Sentiment Analysis (Binary Classification)
**Student:** Dominic &nbsp;|&nbsp; **RA:** RA2512044015131

## Objective
Predict the **Rating** (0 or 1) of a product review using its `Review_Title` and `Review` text.  
Evaluation metric: **F1-Score (binary)**.

## Approach
| Step | Technique |
|---|---|
| Text features | TF-IDF word n-grams (1–3) + char n-grams (2–5), 350K features |
| Models | Logistic Regression (C=5) + Calibrated LinearSVC (C=0.3) |
| Validation | 5-fold Stratified CV + threshold tuning |
| Pseudo-labeling | 3 rounds at confidence ≥ 0.90 |
| Best OOF F1 | ~0.9934 |
| Kaggle eval F1 | **0.9936** (98.88% accuracy, DeBERTa-v3-large + TF-IDF ensemble) |

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, vstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
print('Libraries loaded')

## 1. Load Data

In [ ]:
BASE  = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

# Adjust paths for Kaggle
if os.path.exists('/kaggle/input'):
    import glob
    TRAIN = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
    TEST  = glob.glob('/kaggle/input/**/test.csv',  recursive=True)[0]
else:
    TRAIN = os.path.join(BASE, 'train (1).csv')
    TEST  = os.path.join(BASE, 'test.csv')

train = pd.read_csv(TRAIN)
test  = pd.read_csv(TEST)

print(f'Train: {train.shape}  |  Test: {test.shape}')
print('\nRating distribution (train):')
print(train['Rating'].value_counts())
print(f"\nClass imbalance ratio: {train['Rating'].value_counts()[1]} positive vs {train['Rating'].value_counts()[0]} negative")
train.head(3)

## 2. Text Preprocessing
- Lowercase + collapse whitespace
- **Title repeated ×3** — titles are short, high-signal summaries, so we upweight them

In [ ]:
def clean(text):
    if not isinstance(text, str):
        return ''
    return re.sub(r'\s+', ' ', text.lower()).strip()

train['text'] = ((train['Review_Title'].fillna('') + ' ') * 3 + train['Review'].fillna('')).apply(clean)
test['text']  = ((test['Review_Title'].fillna('') + ' ')  * 3 + test['Review'].fillna('')).apply(clean)

print('Sample:')
print(train['text'].iloc[0][:200])

## 3. TF-IDF Feature Engineering
- **Word n-grams (1–3):** captures unigrams, bigrams, trigrams; `sublinear_tf` dampens term frequency
- **Char n-grams (2–5):** robust to typos and morphological variation
- Fit on train + test combined (transductive, no leakage since labels aren't used)

In [ ]:
all_texts = pd.concat([train['text'], test['text']], ignore_index=True)

word_tfidf = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 3), sublinear_tf=True,
    max_features=200_000, min_df=1, token_pattern=r'(?u)\b\w+\b'
)
char_tfidf = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True,
    max_features=150_000, min_df=2
)

word_tfidf.fit(all_texts)
char_tfidf.fit(all_texts)

print(f'Word vocab: {len(word_tfidf.vocabulary_):,}  |  Char vocab: {len(char_tfidf.vocabulary_):,}')

def make_features(texts):
    return hstack([word_tfidf.transform(texts), char_tfidf.transform(texts)], format='csr')

X_tr = make_features(train['text'])
X_te = make_features(test['text'])
y    = train['Rating'].values

print(f'Feature matrix shape: {X_tr.shape}')

## 4. Model Training — 5-Fold Stratified OOF
Two models ensembled:
- **Logistic Regression** (C=5, lbfgs): probabilistic, calibrated
- **Calibrated LinearSVC** (C=0.3): margin-based, better on imbalanced data

OOF predictions used to tune classification threshold (instead of fixed 0.5).

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr  = LogisticRegression(C=5, max_iter=2000, solver='lbfgs', n_jobs=-1)
svc = CalibratedClassifierCV(LinearSVC(C=0.3, max_iter=2000), cv=3)

print('Running OOF cross-validation...')
lr_oof  = cross_val_predict(lr,  X_tr, y, cv=skf, method='predict_proba')[:, 1]
svc_oof = cross_val_predict(svc, X_tr, y, cv=skf, method='predict_proba')[:, 1]
ens_oof = (lr_oof + svc_oof) / 2.0

print(f'LR  binary-F1 @ 0.5: {f1_score(y, (lr_oof  >= 0.5).astype(int)):.6f}')
print(f'SVC binary-F1 @ 0.5: {f1_score(y, (svc_oof >= 0.5).astype(int)):.6f}')
print(f'ENS binary-F1 @ 0.5: {f1_score(y, (ens_oof >= 0.5).astype(int)):.6f}')

## 5. Threshold Optimisation
With 87:13 class imbalance, threshold 0.5 is sub-optimal.  
We sweep thresholds and pick the one maximising **weighted F1**.

In [ ]:
best_t, best_f = 0.5, 0.0
for t in np.arange(0.05, 0.95, 0.005):
    score = f1_score(y, (ens_oof >= t).astype(int), average='weighted')
    if score > best_f:
        best_f, best_t = score, t

print(f'Best threshold : {best_t:.3f}')
print(f'OOF weighted-F1: {best_f:.6f}')
print(f'OOF binary-F1  : {f1_score(y, (ens_oof >= best_t).astype(int)):.6f}')
print()
print(classification_report(y, (ens_oof >= best_t).astype(int)))

## 6. Train on Full Data + Iterative Pseudo-labeling
Pseudo-labeling adds high-confidence test predictions back as training data,  
improving generalisation over 3 iterative rounds.

In [ ]:
lr.fit(X_tr, y)
svc.fit(X_tr, y)
p_te = (lr.predict_proba(X_te)[:, 1] + svc.predict_proba(X_te)[:, 1]) / 2

CONF      = 0.90
PL_ROUNDS = 3

print(f'Pseudo-labeling: {PL_ROUNDS} rounds, confidence = {CONF}')
for rd in range(1, PL_ROUNDS + 1):
    mask = (p_te >= CONF) | (p_te <= 1 - CONF)
    if mask.sum() == 0:
        print(f'  Round {rd}: no confident pseudo-labels, stopping.')
        break
    y_pl  = (p_te[mask] >= CONF).astype(int)
    X_aug = vstack([X_tr, X_te[mask]])
    y_aug = np.concatenate([y, y_pl])

    lr2  = LogisticRegression(C=5, max_iter=2000, solver='lbfgs', n_jobs=-1)
    svc2 = CalibratedClassifierCV(LinearSVC(C=0.3, max_iter=2000), cv=3)
    lr2.fit(X_aug, y_aug)
    svc2.fit(X_aug, y_aug)
    p_te = (lr2.predict_proba(X_te)[:, 1] + svc2.predict_proba(X_te)[:, 1]) / 2

    p_tr = (lr2.predict_proba(X_tr)[:, 1] + svc2.predict_proba(X_tr)[:, 1]) / 2
    print(f'  Round {rd}: pseudo={mask.sum():5d} | '
          f'train binary-F1={f1_score(y,(p_tr>=best_t).astype(int)):.6f}')

## 7. Generate & Save Submission

In [ ]:
final_preds = (p_te >= best_t).astype(int)
sub = pd.DataFrame({'ID': test['ID'], 'Rating': final_preds})

OUT = '/kaggle/working/submission.csv' if os.path.exists('/kaggle') else os.path.join(os.getcwd(), 'submission.csv')
sub.to_csv(OUT, index=False)

print(f'Saved: {OUT}')
print('\nPrediction distribution:')
print(sub['Rating'].value_counts().to_string())
sub.head()

## Results Summary

| Model | OOF binary F1 | Notes |
|---|---|---|
| LR only (C=5) | ~0.992 | Baseline |
| LR + SVC ensemble | ~0.993 | +0.001 |
| + Pseudo-labeling (3 rounds) | ~0.9934 | +0.0004 |
| DeBERTa-v3-large + TF-IDF (Kaggle) | **0.9936** | 98.88% accuracy |

**Hackathon best submission F1: 0.9935107**